# Banglish + Chatgaiya Threat Severity Classifier — ML Lab Training Notebook

I built a classifier that reads a Banglish (romanized Bengali) or Chatgaiya message and predicts its threat severity, from Level 0 (normal) to Level 3 (direct threat). I use classical machine learning only — no deep learning.

| Level | Meaning | Example |
|---|---|---|
| 0 | Normal | `kal class hobe?` |
| 1 | Teasing / mocking | `tui ekta pagol` |
| 2 | Harassment / abuse | `tui ekta beyadob` (no physical threat) |
| 3 | Direct physical threat | `toke marbo`, `rastay paile tore dekhbo` |

Why this problem: most threats and abuse on Facebook and WhatsApp arrive in Banglish, but public datasets are almost all Bangla script. Chatgaiya (Chittagong dialect) abuse does not appear in any public dataset I could find. People also hide abuse with obfuscation (`* # . _ - ,` inside words, e.g. `চু,,, দি`), which a plain keyword list cannot catch.

Method in one line: TF-IDF features (word 1–3 grams + character 2–6 grams) with Logistic Regression, compared against five other classical classifiers. I measure a simple baseline (word features only, no normalization) against the improved model, so the improvement is visible in numbers.

Two tracks, two different jobs. Track A (Sections 1–9) classifies a word or phrase by severity — this is the shipped model behind the live checker. Track B (Section 10) classifies a whole sentence as affection, abuse, or normal: the same word can be either, depending on who says it, so it needs sentence context. Track B is research, not shipped.


In [ ]:
# 0. Setup — install the packages this notebook needs, then fix the
# random seed so every run gives the same split and the same results.
!pip install -q scikit-learn pandas joblib matplotlib pyarrow

import re, os, random, joblib
import pandas as pd
import numpy as np
random.seed(42)
np.random.seed(42)
print("setup ok")


## 1. Loading the dataset

I train on a Parquet file (`banglish_threat_dataset.parquet`, 12,639 rows) and an optional slang lexicon (`banglish_slang_lexicon.parquet`, 4,269 terms). Parquet is columnar, so I read only the columns I need, and every column has a fixed type. The cell below looks for the files next to the notebook, in Colab's `/content` folder, or in Drive — otherwise it asks me to upload them.

Sources: TB-OLID (5,000 real Banglish comments, CC BY 4.0; offensive/non-offensive labels mapped to severity levels by a lexicon scan), Nirmol v1 (Bangla phrases), LDNOOBW (English bad words, CC BY 4.0), and my own curated Banglish/Chatgaiya examples plus templates. My collection rule: public comments and pages only — no names, phone numbers, or private chats.


In [ ]:
# 1. Load the dataset (Parquet). I look in three places — next to the
# notebook, Colab's /content folder, and Drive — and ask for an upload if
# the file is nowhere to be found.
PARQUET_CANDIDATES = [
    "banglish_threat_dataset.parquet",
    "/content/banglish_threat_dataset.parquet",
    "/content/drive/MyDrive/ML-Lab-Threat/banglish_threat_dataset.parquet",
]
parquet_path = next((p for p in PARQUET_CANDIDATES if os.path.exists(p)), None)

if parquet_path is None:
    try:
        from google.colab import files
        print("Parquet file not found — opening an upload window. Upload banglish_threat_dataset.parquet")
        uploaded = files.upload()
        parquet_path = list(uploaded.keys())[0]
    except Exception as e:
        raise FileNotFoundError("Parquet file not found. Upload banglish_threat_dataset.parquet or put it in Drive.") from e

df = pd.read_parquet(parquet_path)
print("Parquet:", parquet_path, "| rows:", len(df))
print(df.head(10))
print("\n--- class count ---")
print(df["level"].value_counts().sort_index())
print("\n--- dialect count ---")
print(df["dialect"].value_counts())
print("\n--- source count ---")
print(df["source"].value_counts())
print("\n--- dtypes (Parquet schema theke asha) ---")
print(df.dtypes)
print("\n--- missing / duplicate ---")
print("missing text:", df["text"].isna().sum(), "| duplicate rows:", df.duplicated(subset=["text"]).sum())
df = df.dropna(subset=["text"]).drop_duplicates(subset=["text"]).reset_index(drop=True)
print("clean rows:", len(df))

# 1b. Lexicon (optional) — check the slang list by severity, dialect, and source.
LEX_CANDIDATES = [
    "banglish_slang_lexicon.parquet",
    "/content/banglish_slang_lexicon.parquet",
    "/content/drive/MyDrive/ML-Lab-Threat/banglish_slang_lexicon.parquet",
]
lex_path = next((p for p in LEX_CANDIDATES if os.path.exists(p)), None)
if lex_path:
    lex = pd.read_parquet(lex_path)
    print("\n--- lexicon:", lex_path, "| rows:", len(lex), "---")
    print("by severity:\n", lex["severity"].value_counts().sort_index())
    print("by dialect:\n", lex["dialect"].value_counts())
    print("by source:\n", lex["source"].value_counts())
else:
    print("\n(Lexicon file not found — the dataset alone is enough; the lexicon is optional)")

# 1c. Track B — sentence dataset (separate file, separate procedure — Section 10)
SENT_CANDIDATES = [
    "banglish_sentence_dataset.parquet",
    "/content/banglish_sentence_dataset.parquet",
    "/content/drive/MyDrive/ML-Lab-Threat/banglish_sentence_dataset.parquet",
]
sent_path = next((p for p in SENT_CANDIDATES if os.path.exists(p)), None)
if sent_path:
    sdf = pd.read_parquet(sent_path)
    sdf = sdf.dropna(subset=["text"]).drop_duplicates(subset=["text"]).reset_index(drop=True)
    print("\n--- Track B sentence dataset:", sent_path, "| rows:", len(sdf), "---")
    print(sdf["level"].value_counts().sort_index())
    print("sources:\n", sdf["source"].value_counts())
else:
    sdf = None
    print("\n(No Track B sentence file — Track A still runs; upload banglish_sentence_dataset.parquet to see Track B)")


## 2. Normalization — cleaning obfuscated text

People insert `* # . _ - ,` inside abusive words so a machine misses them (`m*rbo`, `চু,,, দি`). My `normalize_text` function strips digits hidden inside Bangla letters (leet), removes mask characters inside words, turns leftover mask characters into spaces, and collapses repeated letters. Normalization is not perfect — new masking styles keep appearing — so Section 6 measures it on a masked test set instead of assuming it works.


In [ ]:
# 2. Normalization — textnorm.py (project-er single source of truth)
LEET = str.maketrans({'0': 'o', '1': 'i', '3': 'e', '4': 'a', '5': 's', '7': 't', '@': 'a', '$': 's'})

def normalize_text(s: str) -> str:
    # Digits hidden inside Bangla letters are leet (ক্রি1ম1নাল -> ক্রিমনাল).
    # Remove them BEFORE the leet mapping, or 1 becomes i and the scripts mix.
    s = re.sub(r"(?<=[\u0980-\u09FF])[0-9]+(?=[\u0980-\u09FF])", "", str(s))
    s = str(s).lower().translate(LEET)
    # Strip mask characters inside words: m*rbo -> mrbo, kh*n -> khn,
    # চু,দি -> চুদি. Commas count as mask characters (Nov 2026: "চু,,, দি").
    s = re.sub(r"(?<=\w)[*#._\-,]+(?=\w)", "", s)
    # Turn remaining mask characters into spaces ("চু,,, দি" -> "চু দি")
    s = re.sub(r"[*#_,]+", " ", s)
    # Collapse a letter repeated 3+ times down to 2
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["clean"] = df["text"].apply(normalize_text)

# Before vs after — 10 masked examples
demo = df[df["dialect"] == "banglish_masked"][["text", "clean"]].head(10)
print(demo.to_string(index=False))
print("\nFraction of masked rows changed by normalization:", (demo["text"] != demo["clean"]).mean(), "fraction")


## 3. Augmentation — making masked copies for training

Real masked examples are rare, so I create masked copies of clean threat examples (one `*` or `#` inside a long word) and add them to the training set only. The test set is never augmented — I measure on real masked rows instead.


In [ ]:
# 3. Augmentation (training set only). I mask one long word per sentence.
def mask_word(w: str) -> str:
    if len(w) < 4:
        return w
    i = random.randint(1, len(w) - 2)
    return w[:i] + random.choice(["*", "#"]) + w[i+1:]

def make_masked(text: str) -> str:
    words = str(text).split()
    if not words:
        return text
    # Mask one word of length 4 or more.
    cands = [i for i, w in enumerate(words) if len(w) >= 4]
    if not cands:
        return text
    i = random.choice(cands)
    words[i] = mask_word(words[i])
    return " ".join(words)

# A few examples:
for t in ["toke marbo", "tore dekhbo rastay", "tui ekta beyadob"]:
    print(t, "->", make_masked(t))


## 4. Train/test split and the model pipeline

I split 80/20 (stratified, seed 42) and build one pipeline per classifier: TF-IDF features, then the classifier. The improved setup (V4, chosen by the grid in Section 5b) uses word 1–3 grams (whole phrases like `তোর মারে চুদি`) plus character 2–6 grams (spelling patterns, so new spellings and unseen targets still match). The baseline uses word features only, with no normalization — Section 6 compares the two.


In [ ]:
# 4. Split + pipelines
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

LEVEL_NAMES = {0: "0-normal", 1: "1-teasing", 2: "2-harassment", 3: "3-threat"}

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["level"])
print("train:", len(train_df), "| test:", len(test_df))
print("train class count:\n", train_df["level"].value_counts().sort_index())

# --- Augmented training set: add a masked copy of each Level 2/3 row (training only) ---
aug_rows = []
for _, r in train_df[train_df["level"] >= 2].iterrows():
    aug_rows.append({"text": make_masked(r["clean"]), "level": int(r["level"]),
                     "dialect": "augmented_masked", "source": "augmentation", "clean": make_masked(r["clean"])})
# The augmented copies already hold masked text — normalize them too.
aug_df = pd.DataFrame(aug_rows)
if len(aug_df):
    aug_df["clean"] = aug_df["text"].apply(normalize_text)
    train_aug = pd.concat([train_df, aug_df], ignore_index=True)
else:
    train_aug = train_df
print("train + augmented:", len(train_aug), "| augmented rows:", len(aug_df))

# V4 (best in the grid): word 1-3 grams + char_wb 2-6 grams.
# Mark-inclusive token pattern: sklearn's default word pattern drops Bengali
# combining marks, so a word like কুত্তা would split into pieces. This
# pattern keeps the word whole (see research/model-comparison-2026-11-18.md).
word_tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=8000, sublinear_tf=True, min_df=2,
                             token_pattern=r"(?u)[\w\u0980-\u09FF]{2,}")
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 6), max_features=12000, sublinear_tf=True, min_df=2)
word_only = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True, min_df=2)

def make_pipe(clf, improved=True):
    # Clone the vectorizers — one fresh copy per pipeline. Sharing a single
    # fitted vectorizer across pipelines is a bug: fitting one pipeline
    # refits it, so a later fit (e.g. Track B) silently changes this
    # pipeline's features and its predictions collapse.
    if improved:
        feats = FeatureUnion([("word", clone(word_tfidf)), ("char", clone(char_tfidf))])
    else:
        feats = clone(word_only)
    return Pipeline([("tfidf", feats), ("clf", clf)])

models = {
    # C=4 (compared on two splits, Nov 2026): +0.011 to +0.014 accuracy over C=1.
    "LogReg": LogisticRegression(C=4, max_iter=1000, class_weight="balanced"),
    "LinearSVC": LinearSVC(class_weight="balanced"),
    "MultinomNB": MultinomialNB(),
    "ComplementNB": ComplementNB(),
    "SGD-log": SGDClassifier(loss="log_loss", max_iter=1000, class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42),
}


In [ ]:
# 5. Train six classifiers on the improved setup (V4) and compare them
# on the same test set.
from sklearn.metrics import recall_score
results = {}
for name, clf in models.items():
    pipe = make_pipe(clf, improved=True)
    pipe.fit(train_aug["clean"], train_aug["level"])
    pred = pipe.predict(test_df["clean"])
    acc = accuracy_score(test_df["level"], pred)
    l3r = recall_score(test_df["level"], pred, labels=[3], average="macro", zero_division=0)
    results[name] = (pipe, acc)
    print(f"\n===== {name} — accuracy: {acc:.3f} | L3 recall: {l3r:.3f} =====")
    print(classification_report(test_df["level"], pred, target_names=[LEVEL_NAMES[i] for i in sorted(LEVEL_NAMES)], zero_division=0))
    print("confusion matrix (row=asol, col=predict):\n", confusion_matrix(test_df["level"], pred))

# Shipping rule: the checker UI shows a confidence bar for each of the 4
# levels, so only a model with predict_proba can ship (LinearSVC has none —
# turning its decision scores into confidences with a softmax would be
# wrong). LogReg is the default; another probability model only replaces it
# if it beats LogReg by more than 0.01 accuracy.
best_name = "LogReg"
best_acc = results["LogReg"][1]
for n in results:
    if n == "LogReg" or n == "LinearSVC":
        continue
    try:
        results[n][0].predict_proba(test_df["clean"][:2])
    except Exception:
        continue
    if results[n][1] > best_acc + 0.01:
        best_name, best_acc = n, results[n][1]
best_pipe = results[best_name][0]
print(f"\n>>> BEST (ship): {best_name} — accuracy {results[best_name][1]:.3f}")
print("    (LinearSVC scores close on accuracy but gives no probabilities, so it cannot ship — see the full grid in Section 5b.)")


## 10. Track B — sentence-level classification (a separate procedure)

Track A and Track B answer different questions, so they train on different data with different procedures. Track A asks how severe a word or phrase is. Track B reads a whole sentence and decides whether it is affection, abuse, or normal — the same word can be either, depending on who says it (`আমার কুত্তার বাচ্চাটা কই গেল` from a mother is affection; `তুই কুত্তার বাচ্চা` from someone else is abuse).

|  | Track A (word-level) | Track B (sentence-level) |
|---|---|---|
| Question | How severe is this word/phrase? | Is this whole sentence affection, abuse, or normal? |
| Data | 12,639 rows (full dataset) | 21,714 sentence rows |
| Why separate? | Recognizing the word is enough | The same word is affectionate or abusive — context decides |

Track B data: Kothon (7,986 sentences), ChatgaiyyaAlap (3,148 sentences + 1,417 dictionary entries), ChatgaiyyaBench (3,009 vulgarity + 1,582 romanized + 204 lexicon entries; machine labels, so some are wrong), ChattogramSent v2 (2,703 sentences; 59% duplicates ChatgaiyyaAlap), ONUBAD (605 sentences + 787 words + 113 clauses), Gemini Deep Research (138 entries), 22 hand-written paired frames, and Meta AI synthetic sentences (marked synthetic in the methodology).

Honest warning: 97% of the sentence data is Level 0 (21,112 of 21,714), and Level 3 has only 28 rows. Accuracy looks good (0.96) but means little here — macro-F1 and per-class recall are the real measures, and Level 3 recall rests on 6 test rows.


In [ ]:
# 10a. Track B — five classifiers, the same V4 features and the same rules
# (80/20 split, seed 42). Same as Track A: normalization plus training-only
# masked augmentation on Level 2+ rows.
from sklearn.metrics import f1_score, recall_score
if sdf is None:
    print("No Track B data — upload banglish_sentence_dataset.parquet and run again.")
else:
    sdf["clean"] = sdf["text"].apply(normalize_text)
    s_train, s_test = train_test_split(sdf, test_size=0.2, random_state=42, stratify=sdf["level"])
    print("Track B train:", len(s_train), "| test:", len(s_test),
          "| test L3 n =", int((s_test["level"] == 3).sum()), "(small sample — do not trust L3 recall too much)")
    s_aug_rows = []
    for _, r in s_train[s_train["level"] >= 2].iterrows():
        m = make_masked(r["clean"])
        s_aug_rows.append({"text": m, "level": int(r["level"]),
                           "dialect": "augmented_masked", "source": "augmentation",
                           "clean": normalize_text(m)})
    s_train_aug = pd.concat([s_train, pd.DataFrame(s_aug_rows)], ignore_index=True)
    print("train + augmented:", len(s_train_aug))
    s_masked = s_test.copy()
    s_masked["masked_clean"] = s_masked["clean"].apply(lambda t: normalize_text(make_masked(t)))

    s_models = {
        "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "LinearSVC": LinearSVC(class_weight="balanced"),
        "MultinomNB": MultinomialNB(),
        "ComplementNB": ComplementNB(),
        "RandomForest": RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42),
    }
    s_results = {}
    for name, clf in s_models.items():
        pipe = make_pipe(clf, improved=True)
        pipe.fit(s_train_aug["clean"], s_train_aug["level"])
        pred = pipe.predict(s_test["clean"])
        acc = accuracy_score(s_test["level"], pred)
        l3r = recall_score(s_test["level"], pred, labels=[3], average="macro", zero_division=0)
        macc = accuracy_score(s_test["level"], pipe.predict(s_masked["masked_clean"]))
        mf1 = f1_score(s_test["level"], pred, average="macro", zero_division=0)
        s_results[name] = (pipe, acc)
        print(f"\n===== Track B {name} — acc: {acc:.3f} | L3 recall: {l3r:.3f} | masked: {macc:.3f} | macro-F1: {mf1:.3f} =====")
        print(classification_report(s_test["level"], pred, target_names=[LEVEL_NAMES[i] for i in sorted(LEVEL_NAMES)], zero_division=0))

    # Track B baseline: word-only, no norm
    s_base = make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced"), improved=False)
    s_base.fit(s_train["text"], s_train["level"])
    s_base_pred = s_base.predict(s_test["text"])
    print(f"\nTrack B baseline (word-only, no norm): acc = {accuracy_score(s_test['level'], s_base_pred):.3f} | "
          f"L3 recall = {recall_score(s_test['level'], s_base_pred, labels=[3], average='macro', zero_division=0):.3f}")
    print("(Baseline 0.845 — with 97% Level 0, guessing 'all Level 0' alone would score high, but it catches almost no teasing or harassment.)")


## 10b. Track B results — reading the numbers

Accuracy looks strong (0.93–0.98) but it is misleading: with 97% Level 0, saying "everything is Level 0" alone scores 0.97. The honest measures are macro-F1 and per-class recall — how much teasing (L1) and harassment (L2) the model actually catches. The exact numbers are printed in the output of the cell above.

LogReg ships for Track B for the same reason as Track A: it gives probabilities, and it has the best L1 recall (0.51) and L2 recall (0.63) among the probability models. LinearSVC has a higher macro-F1 (0.674), but its L1 recall is 0.37 and it gives no probabilities. The variant test is a word-level test, so I do not judge a sentence model by it — that is the point: Track B cannot replace Track A.

Two honest gaps in the demo (Section 10c): the model still calls a mother's `কুত্তার বাচ্চা` abusive, and an angry `ভইঙ্গা` neutral. Only 22 paired frames exist in 21,714 rows, so the model has not learned speaker context yet. More affection/abuse sentence pairs will close this gap.

Summary, both tracks: Logistic Regression wins in both — not because I changed the algorithm, but because the data and the procedure changed. I tried five classifiers on each track; LogReg was the best probability model in both.


In [ ]:
# 10c. Track B demo — the same word, affection vs abuse (paired frames).
print("NOTE (honest): in the output below you will see the model still gets "
      "the mother's affection frame wrong (Level 2 instead of 0) and the "
      "angry bhoinga frame wrong (Level 0 instead of 2). Only 22 paired "
      "frames exist in 21,714 rows, so the model has not learned speaker "
      "context yet. I keep this demo to show where the model still fails, "
      "not to hide it.")
if sdf is not None:
    s_best = s_results["LogReg"][0]
    def s_predict(text):
        c = normalize_text(text)
        lvl = int(s_best.predict([c])[0])
        proba = s_best.predict_proba([c])[0]
        print(f"{text!r} -> Level {lvl} ({LEVEL_NAMES[lvl]}) | "
              f"proba: {{ {', '.join(f'{LEVEL_NAMES[i]}: {float(p):.2f}' for i, p in enumerate(proba))} }}")
        return lvl
    print("--- affection from a mother, abuse from someone else ---")
    for t in ["আমার কুত্তার বাচ্চাটা কই গেল", "কুত্তার বাচ্চা, আয় ভাত খা",
              "তুই কুত্তার বাচ্চা", "তুই একটা কুত্তার বাচ্চা"]:
        s_predict(t)
    print("\n--- ভইঙ্গা: joking vs angry ---")
    for t in ["আরে ভইঙ্গা, কেমন আছস", "তুই ভইঙ্গা, যা এখান থেকে"]:
        s_predict(t)
    print("\n--- neutral everyday sentences ---")
    for t in ["আইজ ক্লাসত যাইবা না?", "বাজরত যাইর", "কাল আইয়োম তোর বারিত"]:
        s_predict(t)
    # Save the Track B model separately — never mix it with Track A's .pkl.
    joblib.dump({"model": s_best, "levels": LEVEL_NAMES, "best_name": "LogReg"},
                "banglish_sentence_model.pkl")
    print("\nsaved: banglish_sentence_model.pkl (Track B, LogReg)")


## 5b. Model comparison grid — which setup is best, and why

I do not trust one model. I run 6 feature setups × 6 classifiers (RandomForest only on 2 setups, because it is slow) and score each on accuracy, Level 3 recall, masked accuracy, and the 90-item variant test. The printed table below is the result — read it before deciding.

What I look for: the best accuracy is not enough. The shipped model must give probabilities (the checker UI shows a confidence bar), so LinearSVC and RandomForest are out even when they top the variant test. LogReg (V4: word 1–3 grams + char 2–6 grams) wins among probability models on accuracy, Level 3 recall, and masked accuracy — that is why it ships.


In [ ]:
# 5b. Grid: 6 vectorizers x 6 classifiers (RandomForest only on 2 setups — slow).
# Each combination is scored on accuracy, Level 3 recall, masked accuracy,
# and the 90-item variant test below.
from sklearn.base import clone
VECTORIZERS = {
    "V1 w1-2+c3-5": lambda: FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True, min_df=2)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=8000, sublinear_tf=True, min_df=2))]),
    "V2 w1-3+c3-5": lambda: FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 3), max_features=5000, sublinear_tf=True, min_df=2)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=8000, sublinear_tf=True, min_df=2))]),
    "V3 w1-2+c2-5": lambda: FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True, min_df=2)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), max_features=8000, sublinear_tf=True, min_df=2))]),
    "V4 w1-3+c2-6": lambda: FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 3), max_features=8000, sublinear_tf=True, min_df=2)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 6), max_features=12000, sublinear_tf=True, min_df=2))]),
    "V5 char-only": lambda: TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=8000, sublinear_tf=True, min_df=2),
    "V6 word-only": lambda: TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True, min_df=2),
}
# variant test — FULL spec list (variant + demo), read from variant_tests.py
# at build time — never hardcode a subset (goes stale vs the 90/90 ship gate).
VARIANT_TESTS = [('ক্রি1ম1নাল', 1), ('তুই একটা রাজাকার', 2), ('রস করা', 2), ('এস থ্রি এক্স', 2), ('তুই একটা দন', 2), ('বাল', 2), ('তুই একটা পুটকি মারো', 3), ('s3x', 2), ('তুই একটা মাদারবোর্ড', 2), ('tui ekta motherboard', 2), ('তুরে চুদবো ছাএলীগে ওয়েট কর আর কয়দিন তুর টোকাইর মারে চু,,, দি এনসিপির মারে চু,,, দি এটা টোকাইয়ার দেশ না এটা শেখ হাসিনার বাংলাদেশ', 3), ('তুরে চুদবো', 3), ('তোর মারে চুদি', 3), ('তোর বোনরে চুদি', 3), ('তোর নানিরে চুদবো', 3), ('তোকে মেরে ফেলবো', 3), ('তোকে খুন করবো', 3), ('তোকে মারবো', 3), ('চু,,, দি', 3), ('মারে চু,,, দি', 3), ('ছাত্রলীগের মারে চুদি', 3), ('এনসিপির মারে চুদি', 3), ('ছাএলীগের মারে চুদি', 3), ('তুর টোকাইর মারে চুদি', 3), ('tor mare chudi', 3), ('tore chudbo', 3), ('তোর ফুফুরে চুদি', 3), ('তোর মামিরে চুদবো', 3), ('তোর বাপরে চুদি', 3), ('তোরে চুদমু', 3), ('tor fufure chudi', 3), ('m0therboard', 2), ('আমি ছাত্রলীগ করি', 0), ('আমার মারে ভালোবাসি', 0), ('এটা শেখ হাসিনার বাংলাদেশ', 0), ('আমার মাকে ভালোবাসি', 0), ('ওয়েট কর, আসতেছি', 0), ('তোর মারে সালাম দিস', 0), ('এনসিপি ভালো দল', 0), ('ক্ষণিকের পোলা', 2), ('তুই একটা ক্ষণিকের পোলা', 2), ('তুই একটা ক্ষণিকের পোয়া', 2), ('khoniker pola', 2), ('tui ekta khoniker pola', 2), ('খনিকের পোলা', 2), ('আমার পোলা স্কুলে যায়', 0), ('পোলাটা ভালো ছেলে', 0), ('oi pola', 0), ('সুদানির ফুয়া, কেমন আছস?', 1), ('তুই সুদানির ফুয়া', 2), ('সুদির ফুয়া তোরে ফুন মারি দিউম', 3), ('তোরা সুদানির ফুয়া অল', 1), ('কুত্তাটা ঘেউ ঘেউ করে', 0), ('তুই একটা কুত্তা', 2), ('আঁর মগ', 0), ('আঁই গম আছি', 0), ('তুই একটা মাগি', 2), ('তেরা বেগ্গুন খানকির ফুয়া', 2), ('কিরে বোকাচোদা', 1), ('সোনার চেইন', 0), ('তুই ভইঙ্গা', 1), ('আমি চুদারফুয়া', 2), ('বাইনচোদ তুই আর কি', 1), ('আই বাটফার গোলা আরকি', 1), ('সেরা বাজি, কেমন আছস', 0), ('আরার চাটগাঁইয়া অক্কল তো সেরা বাজি', 0), ('গম আছি', 0), ('তুই কুত্তার বাচ্চা', 2), ('আমার কুত্তার বাচ্চাটা কই গেল', 0), ('চুনির ফুয়া', 2), ('অডা তুঁই তো আঁর জানের বন্ধু', 0), ('অডা তোঁয়ার মুখ ন দেখতাম চাই', 1), ('হারামি তুঁই তো আঁর জান', 0), ('হারামি, তুঁই একটা চোর', 1), ('মাগির ফুত, তুঁই আঁর জানের বন্ধু', 2), ('মাগির ফুত, তুঁই একটা জানোয়ার', 2), ('বলদ, তুঁই তো আঁর জানের বন্ধু', 0), ('বলদ, তুঁই একটা গাধা', 1), ('কুত্তা, তুঁই আঁর জানের বন্ধু', 2), ('কুত্তা, তুঁই একটা জানোয়ার', 2), ('বাজিরে, কেমন আছস', 0), ('বাজি, কেমন আছস', 0), ('kal class hobe?', 0), ('tui ekta pagol', 1), ('tui ekta beyadob', 2), ('toke marbo', 3), ('toke m*rbo', 3), ('tore dekhbo rastay', 3), ('tui aai ekbar, tore dekhilom', 3), ('basa chini ami, aitesi', 3)]
masked_test = test_df.copy()
masked_test["masked_clean"] = masked_test["clean"].apply(lambda t: normalize_text(make_masked(t)))
for vname, vmake in VECTORIZERS.items():
    for cname, cmake in models.items():
        if cname == "RandomForest" and vname not in ("V1 w1-2+c3-5", "V4 w1-3+c2-6"):
            continue
        # models dict holds INSTANCES (not factories) — clone each one;
        # cmake() raises TypeError (2026-11-18: grid cell never ran).
        pipe = Pipeline([("tfidf", vmake()), ("clf", clone(cmake))])
        pipe.fit(train_aug["clean"], train_aug["level"])
        pred = pipe.predict(test_df["clean"])
        acc = accuracy_score(test_df["level"], pred)
        l3r = recall_score(test_df["level"], pred, labels=[3], average="macro", zero_division=0)
        macc = accuracy_score(masked_test["level"], pipe.predict(masked_test["masked_clean"]))
        vpred = pipe.predict([normalize_text(t) for t, _ in VARIANT_TESTS])
        vscore = sum(int(p == e) for p, (_, e) in zip(vpred, VARIANT_TESTS))
        print(f"{vname:14s} {cname:12s} acc={acc:.3f} L3r={l3r:.3f} masked={macc:.3f} variant={vscore}/{len(VARIANT_TESTS)}")


In [ ]:
# 6. Baseline vs improved — the number that matters for the report.
# Baseline: word features only, raw text, no normalization, no augmentation.
# Improved: word + char features, normalized text, augmented training set.
# Baseline first, then the improved model on the same test set.
base_pipe = make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced"), improved=False)
base_pipe.fit(train_df["text"], train_df["level"])
base_pred = base_pipe.predict(test_df["text"])
imp_pred = best_pipe.predict(test_df["clean"])
print(f"Baseline (word-only, no norm):  accuracy = {accuracy_score(test_df['level'], base_pred):.3f}")
print(f"Improved (word+char, norm+aug): accuracy = {accuracy_score(test_df['level'], imp_pred):.3f}")
print("\n--- Level 3 (direct threat) recall ---")
from sklearn.metrics import recall_score
print("baseline L3 recall:", recall_score(test_df["level"], base_pred, labels=[3], average="macro", zero_division=0))
print("improved L3 recall:", recall_score(test_df["level"], imp_pred, labels=[3], average="macro", zero_division=0))

# --- Masked-only test: how each model scores on the masked test set ---
masked_test = test_df.copy()
masked_test["masked_text"] = masked_test["clean"].apply(make_masked)
masked_test["masked_clean"] = masked_test["masked_text"].apply(normalize_text)
base_masked = base_pipe.predict(masked_test["masked_text"])
imp_masked = best_pipe.predict(masked_test["masked_clean"])
imp_raw_masked = best_pipe.predict(masked_test["masked_text"])  # normalization chara improved model
print("\n--- Masked test set accuracy ---")
print("baseline on masked text:        ", round(accuracy_score(masked_test["level"], base_masked), 3))
print("improved WITHOUT normalization:", round(accuracy_score(masked_test["level"], imp_raw_masked), 3))
print("improved WITH normalization:   ", round(accuracy_score(masked_test["level"], imp_masked), 3))
print("(Read the three masked numbers together: they show whether normalization itself helps on masked text, apart from the other improvements.)")


## 7. Explainability — why did the model say this level?

I read the strongest word and character patterns per level from the Logistic Regression coefficients. If someone asks why a message got Level 3, this list is the answer.


In [ ]:
# 7. Strongest features per level (Logistic Regression only — other
# classifiers have no coefficients to read).
try:
    logreg_pipe = make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced"), improved=True)
    logreg_pipe.fit(train_aug["clean"], train_aug["level"])
    feats = logreg_pipe.named_steps["tfidf"].get_feature_names_out()
    coefs = logreg_pipe.named_steps["clf"].coef_
    for lvl in sorted(LEVEL_NAMES):
        idx = np.argsort(coefs[lvl])[-10:][::-1]
        print(f"Level {lvl} ({LEVEL_NAMES[lvl]}):", [feats[i] for i in idx])
except Exception as e:
    print("explainability skip:", e)


## 8. Saving the best model and trying it out

I save the best pipeline as a `.pkl` file — the live checker loads this exact file, so what I demo here is what runs in production. The demo below runs the same `predict` function on new messages.


In [ ]:
# 8. Save the best model.
joblib.dump({"model": best_pipe, "levels": LEVEL_NAMES, "best_name": best_name},
            "banglish_threat_model.pkl")
print("saved: banglish_threat_model.pkl  (model:", best_name, ")")
print("file size:", os.path.getsize("banglish_threat_model.pkl"), "bytes")

# 9. Demo — try the saved model on new messages.
def predict(text):
    c = normalize_text(text)
    lvl = int(best_pipe.predict([c])[0])
    proba = None
    try:
        proba = best_pipe.predict_proba([c])[0]
    except Exception:
        pass
    action = {0: "nothing to do", 1: "ignore or block",
              2: "block and report", 3: "block, report, and escalate if needed"}[lvl]
    print(f"input: {text!r} -> Level {lvl} ({LEVEL_NAMES[lvl]}) | action: {action}")
    if proba is not None:
        print("   confidence:", {LEVEL_NAMES[i]: round(float(p), 2) for i, p in enumerate(proba)})
    return lvl

for t in ["kal class hobe?", "tui ekta pagol", "tui ekta beyadob",
          "toke marbo", "toke m*rbo", "tore dekhbo rastay",
          "tui aai ekbar, tore dekhilom", "basa chini ami, aitesi",
          # Family-targeted threats, unseen targets, and neutral counter-examples.
          "তুরে চুদবো", "তোর মারে চুদি", "তোর বোনরে চুদি",
          "তোকে মেরে ফেলবো", "চু,,, দি", "ছাত্রলীগের মারে চুদি",
          "তোর ফুফুরে চুদি", "তোরে চুদমু", "m0therboard",
          "আমি ছাত্রলীগ করি", "আমার মারে ভালোবাসি",
          "ওয়েট কর, আসতেছি", "তোর মারে সালাম দিস"]:
    predict(t)


## 9. Next steps and limitations

1. **More real data.** Direct threats (Level 3) are the rarest class (832 of 12,639 rows), so Level 3 recall rests on few examples. More real anonymized messages — especially Chatgaiya and direct threats — would help most.
2. **Masking is not fully solved.** Masked accuracy is 0.740, and normalization itself does not help the masked set (0.740 with it, 0.755 without). It stays because comma-masking and leet variants cannot be caught without it.
3. **Borderline sentences flip.** A friendly frame like `কুত্তা, তুঁই আঁর জানের বন্ধু` is Level 2 but only just (0.503 vs 0.483 for Level 0) — new data can flip it.
4. **Level 3 stays rare.** Real direct threats are hard to find; I use `class_weight="balanced"` to compensate, and I say so in the report.
5. **TB-OLID labels are mapped, not gold.** TB-OLID only has offensive/non-offensive labels; I mapped them to Levels 1–3 with a lexicon scan, so some rows are mislabeled.
6. **Ethics.** No names, phone numbers, or private chats in the data; the model detects, it does not accuse.
7. **Track B (sentence) limits.** Level 3 has 28 rows (6 in test) — its 0.500 recall is a small-sample number. L1 recall is 0.51. Some sentence labels are machine-made (silver) or synthetic — the methodology says so openly. As more sentence data arrives, Track B is retrained and these numbers change.

Results as of round 14 (18 Nov 2026): 12,639 rows (L0 5,491 / L1 1,870 / L2 4,446 / L3 832), lexicon 4,269 terms. Track A shipped: accuracy 0.803 (baseline 0.688), Level 3 recall 0.939 (baseline 0.762), masked 0.740, variant + demo 90/90. Track B: LogReg macro-F1 0.606 (Section 10).

The notebook trains the same pipeline; its numbers differ slightly from the shipped model (accuracy 0.799 here vs 0.803 shipped, L3 recall 0.946 vs 0.939, masked 0.739 vs 0.740 with normalization and 0.750 vs 0.755 without) because the masked augmentation uses random choices.
